# 03 — Evidence + Stack Ensemble

This notebook (a) validates the three bases' lineage, (b) projects each
base's directional prediction pairs into antisymmetric logit-space evidence
(one observation per physical match), (c) trains a zero-intercept logistic
stacker on the OOF evidence, and (d) logs the candidate run and writes the
manifest + test evidence for 05.

Each 02 notebook saved its own best model's directional OOF and test
predictions (`{class}_oof.parquet`, `{class}_test.parquet`) plus the tuning
metric (`{class}_score.json`). Every artifact now holds TWO rows per physical
match (both orientations); `pred` is P(that row's player_id wins).

This notebook validates lineage across the three bases, picks ONE
deterministic evaluation orientation per match, and projects each base's
directional pair into antisymmetric logit-space evidence:

    evidence = (logit(clip(p_chosen)) - logit(clip(p_other))) / 2

so the stacker sees exactly one observation per physical match and can never
see two independent mirrors. No sklearn artifact path is assumed — the NN
participates through its saved predictions.


In [ ]:
from src.utils import load_env

load_env()

from src.constants import CANDIDATE_MANIFEST, DATA_PROCESSED

input_dir = str(DATA_PROCESSED)

output_dir = str(DATA_PROCESSED)

random_state = 42

model_names = ["linear", "gbdt", "nn"]

candidate_manifest = str(CANDIDATE_MANIFEST)

In [ ]:
import json
from typing import Any
import mlflow
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from src.evaluate.symmetry import antisymmetric_evidence, evidence_to_probability

In [ ]:
# ── Load labels and identity columns ──
info_train = pd.read_parquet(f"{input_dir}/info_train.parquet").reset_index(drop=True)
info_test = pd.read_parquet(f"{input_dir}/info_test.parquet").reset_index(drop=True)
y_train = pd.read_parquet(f"{input_dir}/y_train.parquet")["y"].reset_index(drop=True)
y_test = pd.read_parquet(f"{input_dir}/y_test.parquet")["y"].reset_index(drop=True)
print(f"Train: {len(y_train)} rows ({info_train['match_id'].nunique()} matches)")
print(f"Test:  {len(y_test)} rows ({info_test['match_id'].nunique()} matches)")

In [ ]:
# ── Directional base predictions -> antisymmetric evidence (one row per match) ──
# Each 02 artifact has TWO rows per physical match (both orientations). For
# every match we pick ONE deterministic evaluation orientation — the row
# whose player_id is lexicographically smaller — and project the pair into
# logit space. The evidence is in the chosen orientation, so the ensemble
# and its evaluation never double-count a match.
names = list(model_names)
ID_COLS = ["match_id", "player_id", "opponent_id"]


def _load_bases(kind):
    """Load the three bases' directional {kind} predictions and validate that
    all three share identical identity (and OOF fold) sequences."""
    frames = {}
    for name in names:
        df = pd.read_parquet(f"{input_dir}/{name}_{kind}.parquet")
        expected = ID_COLS + (["fold"] if kind == "oof" else []) + ["pred"]
        if list(df.columns) != expected:
            raise ValueError(
                f"{name}_{kind}.parquet columns must be {expected}, got {list(df.columns)}"
            )
        counts = df["match_id"].value_counts()
        bad = int((counts != 2).sum())
        if bad:
            raise ValueError(
                f"{name}_{kind}.parquet: {bad} match_id(s) do not appear exactly twice"
            )
        frames[name] = df
    ref = frames[names[0]]
    for name in names[1:]:
        for col in ID_COLS + (["fold"] if kind == "oof" else []):
            if not (frames[name][col].astype(str) == ref[col].astype(str)).all():
                raise ValueError(f"{name}_{kind} {col} sequence differs from {names[0]}_{kind}")
    return frames


def _assert_mirrored(df: pd.DataFrame, label: str) -> None:
    """Each match_id's two rows must be swapped orientations (A,B)/(B,A)."""
    frame: Any = df
    pairs_by_match = {}
    for raw_row in pd.DataFrame.to_dict(frame, orient="records"):
        row: Any = raw_row
        pairs_by_match.setdefault(row.get("match_id"), []).append(
            (str(row.get("player_id")), str(row.get("opponent_id")))
        )

    for match_id, rows in pairs_by_match.items():
        first, second = rows
        first_player, first_opponent = first
        second_player, second_opponent = second
        if not ((first_player, first_opponent) == (second_opponent, second_player)):
            raise ValueError(
                f"{label}: match {match_id} rows are not mirrored orientations: {rows}"
            )


def _evidence_and_eval(frames, labels, kind):
    """One antisymmetric evidence row per physical match in the chosen
    orientation (lexicographically smaller player_id), plus that row's label."""
    ref = frames[names[0]].reset_index(drop=True)
    chosen_mask = (ref["player_id"].astype(str) <= ref["opponent_id"].astype(str)).to_numpy()
    n_matches = ref["match_id"].nunique()
    if chosen_mask.sum() != n_matches:
        raise ValueError(
            f"{kind}: expected exactly one chosen row per match, "
            f"got {chosen_mask.sum()} for {n_matches} matches"
        )
    match_ids = ref.loc[chosen_mask, "match_id"].to_numpy()
    y_chosen = labels.iloc[np.flatnonzero(chosen_mask)].to_numpy()
    order = np.argsort(match_ids, kind="stable")
    evidence = {}
    for name in names:
        df = frames[name].reset_index(drop=True)
        if not (
            df.loc[chosen_mask, "match_id"].to_numpy()
            == df.loc[~chosen_mask, "match_id"].to_numpy()
        ).all():
            raise ValueError(f"{name}_{kind}: chosen/other rows are not paired per match")
        p_chosen = df.loc[chosen_mask, "pred"].to_numpy()
        p_other = df.loc[~chosen_mask, "pred"].to_numpy()
        evidence[name] = np.asarray(antisymmetric_evidence(p_chosen, p_other))[order]
    eval_df = pd.DataFrame({"match_id": match_ids[order], "match_won": y_chosen[order]})
    return pd.DataFrame(evidence), eval_df


for kind, info, labels in (("oof", info_train, y_train), ("test", info_test, y_test)):
    frames = _load_bases(kind)
    for name in names:
        _assert_mirrored(frames[name], f"{name}_{kind}")
    info_ref = info[ID_COLS].astype(str).to_numpy()
    for name in names:
        if not (frames[name][ID_COLS].astype(str).to_numpy() == info_ref).all():
            raise ValueError(f"{name}_{kind} identity sequence differs from info_{kind} order")
    evidence, eval_df = _evidence_and_eval(frames, labels, kind)
    if kind == "oof":
        oof_evidence, oof_eval = evidence, eval_df
        print(f"oof:  {len(eval_df)} matches -> evidence held in memory ({list(evidence.columns)})")
    else:
        test_evidence, test_eval = evidence, eval_df
        evidence.to_parquet(f"{output_dir}/test_evidence.parquet", index=False)
        eval_df.to_parquet(f"{output_dir}/test_eval.parquet", index=False)
        print(
            f"test: {len(eval_df)} matches -> test_evidence.parquet ({list(evidence.columns)}) + test_eval.parquet"
        )

for name in names:
    with open(f"{input_dir}/{name}_score.json") as f:
        score = json.load(f)
    print(f"Best {name:6s}: {score['metric']} = {score['score']:.4f}")

In [ ]:
# ── Consolidate pinned base-model identities from 02 ──
# Each 02 notebook wrote {name}_model_version.json with the name to register
# under at promotion, the run ID, and the run-artifact model URI. The stacker
# logs these pins; 05 registers each base (and the ensemble) only on
# promotion. No version exists yet, so no alias or `latest` resolution happens
# anywhere downstream. Pins stay in memory for the stacking and MLflow cells
# below; nothing is written here.
base_pins = {}
for name in names:
    with open(f"{input_dir}/{name}_model_version.json") as f:
        base_pins[name] = json.load(f)
with open(f"{input_dir}/aux_pins.json") as f:
    aux_pins = json.load(f)
with open(f"{input_dir}/similarity_pins.json") as f:
    similarity_pins = json.load(f)
aux_pins = {**aux_pins, **similarity_pins}
for name, pin in base_pins.items():
    print(f"  {name}: {pin['registered_model_name']} (run {pin['run_id'][:8]})")

In [ ]:
# ── Compare best-per-class ROC-AUC (OOF, chosen-orientation evidence) ──
# Each base's symmetric probability sigmoid(evidence) vs the chosen
# orientation label — one observation per physical match.
for name in names:
    score = roc_auc_score(oof_eval["match_won"], evidence_to_probability(oof_evidence[name]))
    print(f"  {name:8s} OOF ROC-AUC: {score:.4f}")

In [ ]:
# ── Train the no-intercept stacker on the antisymmetric OOF evidence ──
# The stacker consumes base evidence columns BY NAME in the fixed order
# linear, gbdt, nn. Missing, extra, duplicate, or reordered columns fail
# loudly here before any model is fitted. A zero-intercept logistic head
# over evidence is exactly antisymmetric: a reversed request negates every
# evidence input (evidence -> -evidence), and w·(-e) = -w·e, so the output
# probability complements exactly.
STACK_ORDER = list(model_names)


def _require_evidence_columns(df, label):
    cols = list(df.columns)
    if len(cols) != len(set(cols)):
        raise ValueError(f"{label}: duplicate evidence columns: {cols}")
    if set(cols) != set(STACK_ORDER):
        raise ValueError(f"{label}: evidence columns must be exactly {STACK_ORDER}, got {cols}")
    return df[STACK_ORDER].reset_index(drop=True)


oof_evidence = _require_evidence_columns(oof_evidence, "oof_evidence")
if list(oof_eval.columns) != ["match_id", "match_won"]:
    raise ValueError(
        f"oof_eval columns must be [match_id, match_won], got {list(oof_eval.columns)}"
    )
if not oof_eval["match_id"].is_unique:
    raise ValueError("oof_eval must hold exactly one row per physical match")
if len(oof_eval) != len(oof_evidence):
    raise ValueError(
        f"oof_eval rows ({len(oof_eval)}) differ from oof_evidence rows ({len(oof_evidence)})"
    )
y_train = oof_eval["match_won"].to_numpy()

meta = LogisticRegression(fit_intercept=False, random_state=random_state)
meta.fit(oof_evidence, y_train)

# Assert the no-intercept contract: the fitted model has a zero intercept.
assert not meta.fit_intercept
assert np.allclose(meta.intercept_, 0.0)

print(f"OOF evidence shape: {oof_evidence.shape} (columns {list(oof_evidence.columns)})")
print("Meta-model coefficients (evidence -> logit):")
coef_values = np.asarray(meta.coef_).reshape(-1).tolist()
for name, coef in zip(STACK_ORDER, coef_values, strict=False):
    print(f"  {name:8s} {coef:.4f}")

In [ ]:
# ── Log pinned candidate to MLflow (no registration/promotion) ──
# base_pins and aux_pins were consolidated in memory above. The pins logged
# here are the exact run artifacts this candidate was trained from, so 05
# tags the promoted version and deploy resolves exactly these runs. The
# bases are registered at promotion time (05). No alias or `latest`
# resolution happens anywhere downstream.
mlflow.set_experiment("stacked_ensemble")
with mlflow.start_run():
    for name in STACK_ORDER:
        pin = base_pins[name]
        mlflow.log_param(f"base_{name}_registered_name", pin["registered_model_name"])
        mlflow.log_param(f"base_{name}_run_id", pin["run_id"])
        mlflow.log_param(f"base_{name}_model_uri", pin["model_uri"])
    mlflow.log_metrics(
        {f"weight_{name}": coef for name, coef in zip(STACK_ORDER, coef_values, strict=False)}
    )
    # The stacker is fitted with fit_intercept=False; log the zero intercept
    # explicitly so the metadata records the antisymmetry contract.
    mlflow.log_metric("intercept", 0.0)
    stacked_info = mlflow.sklearn.log_model(meta, "stacked_ensemble")
    run = mlflow.active_run()
    assert run is not None
    run_id = run.info.run_id
    print(f"Candidate logged to MLflow run: {run_id}")

In [ ]:
# ── Manifest for 05: exact candidate run + full lineage, no latest lookup ──
manifest = {
    "candidate_run_id": run_id,
    "model_uri": stacked_info.model_uri,
    "artifact_path": "stacked_ensemble",
    "base_pins": base_pins,
    "aux_pins": aux_pins,
}
with open(candidate_manifest, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Manifest written to {candidate_manifest}")
print(f"  candidate_run_id: {run_id}")
print(f"  model_uri:        {stacked_info.model_uri}")
print(f"  lineage: {len(base_pins)} base classes, {len(aux_pins)} aux keys")